# Project 1 — Qwen Customer Support Chatbot

**Milestone 1:** run inference locally in Colab, inspect tokens, then launch a simple Gradio chat.

Run these cells from top to bottom. Do not commit private data or model weights.


In [ ]:
%pip -q install "transformers>=4.45,<5" "gradio>=5,<7" accelerate


In [ ]:
import torch
import transformers
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()
print("Model and tokenizer ready on", device)


## Inference core

Chat messages are formatted according to the model's chat template. The returned text is only the **newly generated** tokens. This approach avoids indexing a `BatchEncoding` as though it were a tensor.


In [ ]:
SYSTEM_PROMPT = (
    "You are a demo customer support assistant. Speak clearly and concisely. "
    "Do not invent company policies, prices, or delivery promises. "
    "If unsure, say so and ask a useful question."
)

def generate_reply(message, history=None, max_new_tokens=128):
    if not isinstance(message, str) or not message.strip():
        return "Please enter a question."
    history = history or []
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for item in history[-6:]:
        if (isinstance(item, dict) and item.get("role") in ("user", "assistant")
                and isinstance(item.get("content"), str)):
            messages.append({"role": item["role"], "content": item["content"]})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    batch = tokenizer([prompt], return_tensors="pt").to(device)
    if batch["input_ids"].shape[1] > 1500:
        return "Conversation too long for this demo. Please start a new chat."
    with torch.inference_mode():
        out = model.generate(
            **batch, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = out[0, batch["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


In [ ]:
answer = generate_reply("What is an LLM? Explain in two short sentences.")
print("AI response:", answer)
assert len(answer) > 0, "No text generated"


## Tokenization experiment

These IDs are the numerical representations of the tokens; IDs are model-specific.


In [ ]:
sample = "I need an online store."
ids = tokenizer.encode(sample, add_special_tokens=False)
print("Token count:", len(ids))
print("Token IDs:", ids)
print("Token pieces:", tokenizer.convert_ids_to_tokens(ids))


## Optional: launch Gradio

Only run this after the test above succeeds. In Colab, use `share=True` only for non-sensitive demo data: the link is publicly reachable.


In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=generate_reply,
    title="AI Customer Support Chatbot — Portfolio Prototype",
    description="Qwen2.5-0.5B-Instruct running in Google Colab",
    examples=["What can you help with?", "I want a business website."],
)
demo.launch(share=False, debug=True, show_error=True)
